# 04: Regression Analysis — RQ1
**RQ1:** What factors predict student trust and satisfaction with ChatGPT?  
**Team 5 | March 2026**

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/cleaned/cleaned_data.csv')
print(f"Sample: {len(df):,} students")


Sample: 22,836 students


In [2]:
# ============================================================
# PREPARE REGRESSION DATA
# ============================================================
outcome = 'Q15'  # Trust
predictors = ['usage_intensity', 'capabilities_score', 'ethics_score', 'attitudes_score']

reg_data = df[[outcome] + predictors].dropna()
print(f"Regression sample (complete cases): N = {len(reg_data):,}")
print(f"  ({len(reg_data)/len(df)*100:.1f}% of full dataset)")

y = reg_data[outcome]
X = reg_data[predictors]


Regression sample (complete cases): N = 16,237
  (71.1% of full dataset)


In [3]:
# ============================================================
# MULTICOLLINEARITY CHECK (VIF)
# ============================================================
print("Multicollinearity Check (VIF):")
X_const = sm.add_constant(X)
vif = pd.DataFrame({
    'Variable': predictors,
    'VIF': [variance_inflation_factor(X_const.values, i+1) for i in range(len(predictors))]
})
print(vif.to_string(index=False))
print("\nNote: VIF < 5 acceptable, VIF > 10 problematic")
if (vif['VIF'] > 10).any():
    print("⚠ WARNING: Some VIF values exceed 10")
elif (vif['VIF'] > 5).any():
    print("⚠ Caution: Some VIF values exceed 5")
else:
    print("✓ All VIF values acceptable")


Multicollinearity Check (VIF):
          Variable      VIF
   usage_intensity 1.249664
capabilities_score 1.128445
      ethics_score 1.260624
   attitudes_score 1.107818

Note: VIF < 5 acceptable, VIF > 10 problematic
✓ All VIF values acceptable


In [4]:
# ============================================================
# HIERARCHICAL REGRESSION
# ============================================================

# Model 1: Usage Intensity only
X1 = sm.add_constant(reg_data[['usage_intensity']])
m1 = sm.OLS(y, X1).fit()

print("MODEL 1: Usage Intensity Only")
print(f"  R² = {m1.rsquared:.4f}, Adj R² = {m1.rsquared_adj:.4f}")
print(f"  F = {m1.fvalue:.2f}, p = {m1.f_pvalue:.2e}")
print(f"  usage_intensity: β = {m1.params['usage_intensity']:.4f}, p = {m1.pvalues['usage_intensity']:.4e}")

# Model 2: + Capabilities + Ethics
X2 = sm.add_constant(reg_data[['usage_intensity', 'capabilities_score', 'ethics_score']])
m2 = sm.OLS(y, X2).fit()

print("\nMODEL 2: + Capabilities + Ethical Concerns")
print(f"  R² = {m2.rsquared:.4f}, Adj R² = {m2.rsquared_adj:.4f}")
print(f"  F = {m2.fvalue:.2f}, p = {m2.f_pvalue:.2e}")
print(f"  ΔR² from Model 1: +{m2.rsquared - m1.rsquared:.4f}")
for v in ['usage_intensity', 'capabilities_score', 'ethics_score']:
    print(f"  {v:25s}: β = {m2.params[v]:+.4f}, p = {m2.pvalues[v]:.4e}")

# Model 3: Full model (+ Attitudes)
X3 = sm.add_constant(reg_data[predictors])
m3 = sm.OLS(y, X3).fit()

print("\nMODEL 3: Full Model (+ Attitudes)")
print(m3.summary())


MODEL 1: Usage Intensity Only
  R² = 0.3509, Adj R² = 0.3508
  F = 8775.83, p = 0.00e+00
  usage_intensity: β = 0.8449, p = 0.0000e+00

MODEL 2: + Capabilities + Ethical Concerns
  R² = 0.3545, Adj R² = 0.3544
  F = 2972.25, p = 0.00e+00
  ΔR² from Model 1: +0.0037
  usage_intensity          : β = +0.8150, p = 0.0000e+00
  capabilities_score       : β = -0.0642, p = 2.3055e-14
  ethics_score             : β = +0.0550, p = 3.7498e-07

MODEL 3: Full Model (+ Attitudes)
                            OLS Regression Results                            
Dep. Variable:                    Q15   R-squared:                       0.356
Model:                            OLS   Adj. R-squared:                  0.356
Method:                 Least Squares   F-statistic:                     2245.
Date:                Mon, 09 Mar 2026   Prob (F-statistic):               0.00
Time:                        00:08:31   Log-Likelihood:                -21507.
No. Observations:               16237   AIC:          

In [5]:
# ============================================================
# MODEL COMPARISON
# ============================================================
print("Model Comparison:")
comp = pd.DataFrame({
    'Model': ['1: Usage', '2: + Cap + Ethics', '3: Full'],
    'R²': [m1.rsquared, m2.rsquared, m3.rsquared],
    'Adj R²': [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj],
    'ΔR²': [m1.rsquared, m2.rsquared - m1.rsquared, m3.rsquared - m2.rsquared],
    'F': [m1.fvalue, m2.fvalue, m3.fvalue],
    'N': [int(m1.nobs), int(m2.nobs), int(m3.nobs)],
})
print(comp.round(4).to_string(index=False))


Model Comparison:
            Model     R²  Adj R²    ΔR²         F     N
         1: Usage 0.3509  0.3508 0.3509 8775.8296 16237
2: + Cap + Ethics 0.3545  0.3544 0.0037 2972.2484 16237
          3: Full 0.3562  0.3560 0.0016 2244.8320 16237


In [6]:
# ============================================================
# FINAL MODEL COEFFICIENTS
# ============================================================
print(f"\nFinal Model Coefficients:")
print(f"R² = {m3.rsquared:.4f}, Adj R² = {m3.rsquared_adj:.4f}")
print(f"F = {m3.fvalue:.2f}, p < .001, N = {int(m3.nobs):,}")
print(f"\nCoefficients:")
for v in predictors:
    coef = m3.params[v]
    p = m3.pvalues[v]
    ci = m3.conf_int().loc[v]
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    direction = "↑" if coef > 0 else "↓"
    print(f"  {v:25s}: β = {coef:+.4f} {sig}  95%CI [{ci[0]:+.4f}, {ci[1]:+.4f}] {direction}")



Final Model Coefficients:
R² = 0.3562, Adj R² = 0.3560
F = 2244.83, p < .001, N = 16,237

Coefficients:
  usage_intensity          : β = +0.8146 ***  95%CI [+0.7949, +0.8342] ↑
  capabilities_score       : β = -0.0467 ***  95%CI [-0.0640, -0.0294] ↓
  ethics_score             : β = +0.0586 ***  95%CI [+0.0374, +0.0798] ↑
  attitudes_score          : β = -0.0570 ***  95%CI [-0.0745, -0.0395] ↓


In [7]:
# ============================================================
# SAVE RESULTS
# ============================================================
import os
os.makedirs('results/tables', exist_ok=True)
os.makedirs('results/statistical_outputs', exist_ok=True)

comp.to_csv('results/tables/model_comparison.csv', index=False)

with open('results/statistical_outputs/regression_full.txt', 'w') as f:
    f.write("HIERARCHICAL REGRESSION — PREDICTING TRUST (Q15)\n")
    f.write("="*70 + "\n\n")
    f.write("MODEL 1: Usage Intensity\n" + "-"*70 + "\n")
    f.write(m1.summary().as_text() + "\n\n")
    f.write("MODEL 2: + Capabilities + Ethics\n" + "-"*70 + "\n")
    f.write(m2.summary().as_text() + "\n\n")
    f.write("MODEL 3: Full Model\n" + "-"*70 + "\n")
    f.write(m3.summary().as_text())

print("✓ Regression analysis complete — results saved")


✓ Regression analysis complete — results saved
